# Lab 3B: Fine-tune Llama 3.2 3B with Experiment Tracking

## Overview

In this lab, you fine-tune Llama 3.2 3B for text summarization using the
**SageMaker Python SDK v3**. The workflow uses `JumpStartConfig` as the model
contract, `ModelTrainer` for managed fine-tuning, `ModelBuilder` for both base
and fine-tuned deployment, and typed `Endpoint.invoke()` for inference.

The MLflow run records the dataset lineage, training job, model artifact,
inference image, environment, and endpoint names consumed by Lab 3C.


## Step 1: Setup and Install Dependencies

First, you'll install the required libraries and initialize our SageMaker session. This establishes the execution context for our training job.

<div style="padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404;">
<strong>⚠️ Important:</strong> The cell below installs libraries and restarts the kernel. After the restart, continue with the next cell.
</div>

In [ ]:
import importlib.metadata as metadata
from packaging.version import Version
import datasets

sdk_version = Version(metadata.version("sagemaker"))
datasets_version = Version(datasets.__version__)
print(f"SageMaker SDK version: {sdk_version}")
print(f"datasets version:      {datasets_version}")

assert Version("3") <= sdk_version < Version("4"), (
    f"This notebook requires sagemaker>=3,<4; found {sdk_version}. "
    "Run the installation cell and wait for the kernel restart."
)
assert datasets_version >= Version("4.4.1"), (
    f"This notebook requires datasets>=4.4.1; found {datasets_version}."
)
print("✓ Dependency versions are compatible")


In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path

import boto3
import mlflow

from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.jumpstart.configs import JumpStartConfig
from sagemaker.core.training.configs import Compute, InputData, OutputDataConfig
from sagemaker.core.resources import Endpoint
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.serve.model_builder import ModelBuilder

# SDK v3 session and execution role.
sess = Session()
role = get_execution_role()
region = sess.boto_region_name
bucket = sess.default_bucket()  # Same contract as Lab 2A.
account_id = boto3.client("sts", region_name=region).get_caller_identity()["Account"]
sm_client = boto3.client("sagemaker", region_name=region)
s3_client = boto3.client("s3", region_name=region)

print(f"Amazon SageMaker role: {role}")
print(f"Account ID:            {account_id}")
print(f"Amazon S3 bucket:      {bucket}")
print(f"AWS Region:            {region}")


## Step 2: Deploy the Base Model with SDK v3


Next, you will deploy the base Llama 3.2 3B model so that you can later compare its performance with the fine-tuned model for your summarization use case

In [ ]:
# Pin the existing 1.x model contract. The current SDK v3 Hub API rejects the
# old wildcard "1.*", and JumpStart 2.0.0 may have different input/output
# signatures. Pinning 1.1.9 keeps this lab reproducible.
model_id = "meta-textgeneration-llama-3-2-3b"
model_version = "1.1.9"
inference_instance_type = "ml.g5.2xlarge"

jumpstart_config = JumpStartConfig(
    model_id=model_id,
    model_version=model_version,
    accept_eula=True,
)


<div style="background-color: #d4edda; border: 1px solid #c3e6cb; border-radius: 4px; padding: 12px; margin: 10px 0;">
<b>✓ Llama Model EULA Acceptance</b><br>
To deploy Llama models using SageMaker JumpStart, you must accept Meta's End User License Agreement (EULA). In the notebook, set <code>accept_eula=true</code> in the estimator configuration. By doing so, you acknowledge that you have read and agree to the terms of the EULA, available at https://ai.meta.com/resources/models-and-libraries/llama-downloads/. Deployment will fail if this parameter is not set to true.
</div>


> **⏱️ Note:** The deployment job will take approximately 10 minutes to complete.


In [ ]:
# Configure the base JumpStart deployment with SDK v3 ModelBuilder.
base_model_builder = ModelBuilder.from_jumpstart_config(
    jumpstart_config=jumpstart_config,
    role_arn=role,
    compute=Compute(instance_type=inference_instance_type, instance_count=1),
    sagemaker_session=sess,
)

base_endpoint_requested_name = f"llama-3-2-3b-base-{datetime.now().strftime('%Y%m%d%H%M%S')}"
base_model_builder.build(model_name=f"{base_endpoint_requested_name}-model")
print(f"Deploying base model to {base_endpoint_requested_name} (about 10 minutes)...")
base_model_endpoint = base_model_builder.deploy(
    endpoint_name=base_endpoint_requested_name,
    wait=True,
)
assert isinstance(base_model_endpoint, Endpoint)


In [ ]:
from IPython.display import Markdown, display

base_model_endpoint_name = base_model_endpoint.endpoint_name
console_url = (
    f"https://console.aws.amazon.com/sagemaker/home?region={region}"
    f"#/endpoints/{base_model_endpoint_name}"
)
display(Markdown(
    f"You can **[view the base-model endpoint in the SageMaker Console]({console_url})**"
))


In [ ]:
print(base_model_endpoint_name)
%store base_model_endpoint_name


## Step 3: Initialize MLflow for Experiment Tracking

### Understanding SageMaker Managed MLflow

SageMaker Serverless MLflow provides a **fully managed experiment MLflow App** that eliminates the need to set up and maintain your own MLflow infrastructure. Key benefits:

- **Centralized Tracking**: All team members log to the same MLflow App
- **No Infrastructure Management**: AWS handles scaling, backups, and availability
- **Integrated Security**: Uses IAM for authentication and authorization
- **Persistent Storage**: Experiments are stored durably in AWS-managed storage

### What Gets Tracked
When you log experiments to MLflow, you capture:
- **Parameters**: Hyperparameters, model IDs, instance types
- **Metrics**: Training loss, validation accuracy, custom metrics
- **Artifacts**: Model files, training datasets, configuration files
- **Metadata**: Run names, timestamps, tags, notes

This creates a **complete audit trail** for governance and compliance.

If you are running this lab as part of an AWS workshop, an MLFlow App has already been created for you. You can use this App to track your fine-tuning experiments. Let's retrieve the MLFlow App URI, you will use it to track the experiments.

If you are running this notebook in your own environment, you need to have an existing running MLFlow App to be able to complete it successfully. Refer to the [AWS Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html) for more details.

Now you are ready to set up your experiment

In [ ]:
try:
    response = sm_client.list_mlflow_apps(MaxResults=10)
    mlflow_apps = response.get('Summaries', [])
    
    if mlflow_apps:
        active_apps = [app for app in mlflow_apps if app['Status'] in ('Created', 'Updated')]
        
        if active_apps:
            mlflow_app_arn = active_apps[0]['Arn']
            mlflow_app_name = active_apps[0]['Name']
            print(f"✓ Found active MLflow App:")
            print(f"  Name: {mlflow_app_name}")
            print(f"  ARN: {mlflow_app_arn}")
        else:
            raise RuntimeError("No healthy MLflow App found (expected status Created or Updated)")
    else:
        raise RuntimeError("No MLflow Apps found in this region")
except Exception as e:
    raise RuntimeError(f"Could not resolve the workshop MLflow App: {e}") from e

In [ ]:
import mlflow
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

# Connect to the managed MLflow App
mlflow.set_tracking_uri(mlflow_app_arn)

# Create or use existing experiment
# Experiments group related runs together (e.g., all summarization model iterations)
experiment_name = f"summarization-experiment-{timestamp}"
mlflow.set_experiment(experiment_name)

print(f"✓ MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"✓ Experiment: {experiment_name}")

### 📊 View Your Experiment in MLflow

**To access the MLflow UI:**

1. In the left sidebar of SageMaker Studio, click the **MLflow** icon
2. Click on your MlFlow App name
3. Using the Menu on the right hand side, open MLFlow. (See screenshot below)
4. Navigate to your experiment (See screenshot below. The exact experiment name will differ depending on the timestamp) 
![MLFlow Experiment](../../images/mlflow-console.png)
![MLFlow Experiment](../../images/mlflow-experiment.png)

Next initiate a Run in your experiment to track the fine-tuning job.

In [ ]:
# Start a new MLflow run to track this fine-tuning job
# A "run" represents a single training execution with specific parameters
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

run_name=f"llama-3.2-fine-tuning-summarization-{timestamp}"

mlflow_run = mlflow.start_run(run_name=run_name)
print(f"✓ Started MLflow Run ID: {mlflow_run.info.run_id}")
print(f"  This run will track all parameters, metrics, and artifacts from this training job.")

## Step 4: Load the Prepared Fine-Tuning Dataset from S3

### Data prepared in Lab 2
The fine-tuning dataset is now prepared by a standalone notebook: `lab2-data-prep/fine-tuning-data-prep.ipynb`. That notebook loads the Databricks Dolly 15k dataset, filters for **summarization** examples, splits 70/30 into train/test, creates the instruction prompt `template.json`, and uploads the complete package to S3.

> **Prerequisite**: Run `lab2-data-prep/fine-tuning-data-prep.ipynb` first, using the **same `bucket` value** you set in Step 1. It publishes the dataset to the S3 location referenced below.

### Fine-tuning package (produced by Lab 2)
`s3://<DataBucketName>/<ProfileName>/dolly_dataset/`
- `train.jsonl` - training examples
- `test.jsonl` - held-out evaluation examples
- `template.json` - instruction prompt template

Here we reference those S3 paths and pull the files locally for evaluation and inference formatting.

In [ ]:
# Resolve the same default-bucket/profile contract produced by Lab 2A.
metadata_path = Path('/opt/ml/metadata/resource-metadata.json')
if metadata_path.exists():
    profile_name = json.loads(metadata_path.read_text())['UserProfileName']
else:
    profile_name = os.environ.get('SM_USER_PROFILE_NAME', 'userA')
profile_name = profile_name[0].upper() + profile_name[1:]

data_location = f"s3://{bucket}/{profile_name}/dolly_dataset"
train_path = "train.jsonl"
template_path = "template.json"
evaluation_path = "test.jsonl"
training_input_path = f"{data_location}/{train_path}"
eval_input_path = f"{data_location}/{evaluation_path}"


def download_s3_uri(uri, local_path=None):
    """Download one S3 object with the AWS client used by SDK v3."""
    bucket_name, key = uri.removeprefix("s3://").split("/", 1)
    destination = local_path or Path(key).name
    s3_client.download_file(bucket_name, key, str(destination))
    return str(destination)


# Local copies are used for MLflow lineage, evaluation, and prompt formatting.
download_s3_uri(training_input_path, train_path)
download_s3_uri(eval_input_path, evaluation_path)
download_s3_uri(f"{data_location}/{template_path}", template_path)

print(f"✓ Referenced fine-tuning dataset at: {data_location}")
print(f"  Training input:   {training_input_path}")
print(f"  Evaluation input: {eval_input_path}")


In [ ]:
from datasets import load_dataset

# Load the prompt template (used for inference-time prompt formatting in Step 7)
with open(template_path, "r") as f:
    template = json.load(f)

# Load the held-out test split for base vs. fine-tuned comparison in Step 7
test_dataset = load_dataset("json", data_files=evaluation_path, split="train")

print(f"✓ Template loaded and {test_dataset.num_rows} test examples ready")

In [ ]:
import pandas as pd

print("Sample training example:")
pd.read_json(train_path, lines=True).iloc[0].to_dict()

## Step 5: Log Dataset Lineage to MLflow

The dataset already lives in S3 (uploaded by the Lab 2 data prep notebook). Here we log the training and evaluation datasets as inputs to the active MLflow run so the experiment captures complete data lineage for governance and reproducibility.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore', message='Failed to determine whether UCVolumeDatasetSource')

df_train = pd.read_json(train_path, orient="records", lines=True)
training_data = mlflow.data.from_pandas(df_train, source=training_input_path)
mlflow.log_input(training_data, context="training")

In [ ]:
df_evaluate = pd.read_json(evaluation_path, orient="records", lines=True)
df_evaluate.size
evaluation_data = mlflow.data.from_pandas(df_evaluate, source=eval_input_path)
mlflow.log_input(evaluation_data, context="evaluation")

## Step 6: Configure and Launch Fine-Tuning with `ModelTrainer`

SDK v3 resolves the JumpStart training image, source code, model artifact input,
default environment, and EULA access from the same `JumpStartConfig` used for
base-model deployment.


<div style="background-color: #d4edda; border: 1px solid #c3e6cb; border-radius: 4px; padding: 12px; margin: 10px 0;">
<b>✓ Llama Model EULA Acceptance</b><br>
To deploy Llama models using SageMaker JumpStart, you must accept Meta's End User License Agreement (EULA). In the notebook, set <code>accept_eula=true</code> in the estimator configuration. By doing so, you acknowledge that you have read and agree to the terms of the EULA, available at https://ai.meta.com/resources/models-and-libraries/llama-downloads/. Deployment will fail if this parameter is not set to true.
</div>


In [ ]:
# Fine-tuning configuration. Keep values as strings because JumpStart passes
# them to the training container as hyperparameters.
hyperparameters = {
    "epoch": "2",
    "instruction_tuned": "True",
    "max_input_length": "1024",
}
training_instance_type = "ml.g5.2xlarge"
training_output_path = f"s3://{bucket}/{profile_name}/model-output/"

mlflow.log_param("base_model_id", model_id)
mlflow.log_param("model_version", model_version)
for key, value in hyperparameters.items():
    mlflow.log_param(key, value)
print("✓ Hyperparameters configured and logged to MLflow")


In [ ]:
# Configure JumpStart fine-tuning with SDK v3 ModelTrainer.
model_trainer = ModelTrainer.from_jumpstart_config(
    jumpstart_config=jumpstart_config,
    role=role,
    compute=Compute(instance_type=training_instance_type, instance_count=1),
    output_data_config=OutputDataConfig(s3_output_path=training_output_path),
    hyperparameters=hyperparameters,
    sagemaker_session=sess,
    base_job_name="llama-3-2-3b-summarization",
)

mlflow.log_param("instance_type", training_instance_type)
mlflow.log_param("instance_count", 1)
mlflow.log_param("output_path", training_output_path)
mlflow.log_param("training_image_uri", model_trainer.training_image)
print("✓ ModelTrainer configured")
print(f"  Training image: {model_trainer.training_image}")


### Start the training job

In [ ]:
from IPython.display import Markdown, display

console_url = f"https://console.aws.amazon.com/sagemaker/home?region={region}#/training"

display(Markdown(f"You are now ready to start the training job and fine-tune the model. You can review the metadata and progress of the training job in the AWS console **[🔗 View Training Jobs in SageMaker Console]({console_url})**"))


> **⏱️ Note:** The training job will take approximately 15-18 minutes to complete.

In [ ]:
# The training channel must point at the DIRECTORY, not only train.jsonl:
# JumpStart instruction tuning also needs template.json beside the training file.
training_data = InputData(channel_name="training", data_source=data_location)

print("🚀 Starting fine-tuning job (approximately 15-18 minutes)...")
model_trainer.train(
    input_data_config=[training_data],
    wait=True,
    logs=True,
)

training_job = model_trainer._latest_training_job
training_job_name = training_job.training_job_name
model_artifact_s3 = training_job.model_artifacts.s3_model_artifacts

print("\n✓ Fine-tuning job completed")
print(f"  Training job:  {training_job_name}")
print(f"  Model artifact: {model_artifact_s3}")
mlflow.log_param("training_job_name", training_job_name)
mlflow.log_param("model_artifact_s3", model_artifact_s3)


###  Track the training progress
While waiting, you can track the training progress above and also review the information you have logged in MLFLow:
1. Navigate to the MLFlow console
2. Find the summarization - experiment you created earlier
3. Click on its name to view the experiment details
4. Locate the Run and click on its name to view its details

![MLFlow Experiment](../../images/run.png)
![MLFlow Experiment](../../images/run_details.png)

In [ ]:
# Record training output immediately; Step 7 adds the inference image and
# environment after ModelBuilder resolves them during deployment.
mlflow.log_dict(
    {
        "model_artifact": model_artifact_s3,
        "training_job_name": training_job_name,
        "base_model_id": model_id,
        "base_model_version": model_version,
    },
    "training_info.json",
)


In [ ]:
from IPython.display import Markdown, display

s3_path = model_artifact_s3.removeprefix("s3://").split("/", 1)
console_url = f"https://s3.console.aws.amazon.com/s3/object/{s3_path[0]}?prefix={s3_path[1]}"
display(Markdown(
    f"**Training output:** [{model_artifact_s3}]({console_url})"
))


## Step 7: Deploy the Fine-Tuned Model with `ModelBuilder`


> **⏱️ Note:** The deployment job will take approximately 10 minutes to complete.

In [ ]:
# ModelBuilder recognizes a JumpStart ModelTrainer, takes the completed job's
# model artifact, and resolves the matching JumpStart inference image.
fine_tuned_endpoint_requested_name = (
    f"llama-3-2-3b-finetuned-{datetime.now().strftime('%Y%m%d%H%M%S')}"
)
fine_tuned_model_builder = ModelBuilder(
    model=model_trainer,
    role_arn=role,
    compute=Compute(instance_type=inference_instance_type, instance_count=1),
    env_vars={"OPTION_MAX_MODEL_LEN": "65536"},
    sagemaker_session=sess,
)

fine_tuned_model_builder.build(model_name=f"{fine_tuned_endpoint_requested_name}-model")
print(f"Deploying fine-tuned model to {fine_tuned_endpoint_requested_name}...")
fine_tuned_model_endpoint = fine_tuned_model_builder.deploy(
    endpoint_name=fine_tuned_endpoint_requested_name,
    wait=True,
)
assert isinstance(fine_tuned_model_endpoint, Endpoint)


In [ ]:
fine_tuned_model_endpoint_name = fine_tuned_model_endpoint.endpoint_name
print(fine_tuned_model_endpoint_name)
%store fine_tuned_model_endpoint_name


In [ ]:
# After build/deploy, ModelBuilder holds the resolved inference image and the
# complete JumpStart serving environment. Lab 3C registers exactly these values,
# rather than incorrectly registering the training image.
inference_image_uri = fine_tuned_model_builder.image_uri
inference_environment = dict(fine_tuned_model_builder.env_vars)

assert inference_image_uri, "ModelBuilder did not resolve an inference image"
mlflow.log_param("base_endpoint_name", base_model_endpoint_name)
mlflow.log_param("endpoint_name", fine_tuned_model_endpoint_name)
mlflow.log_param("endpoint_instance_type", inference_instance_type)
mlflow.log_param("inference_image_uri", inference_image_uri)
mlflow.log_dict(
    {
        "model_artifact": model_artifact_s3,
        "training_job_name": training_job_name,
        "inference_image_uri": inference_image_uri,
        "inference_environment": inference_environment,
        "base_model_id": model_id,
        "base_model_version": model_version,
    },
    "model_info.json",
)
print(f"✓ Inference image recorded for Lab 3C: {inference_image_uri}")


## Step 8: Compare the Base and Fine-Tuned Models


Let's now do some initial testing to compare the outputs of the base and fine-tuned models

In [ ]:
def invoke_json(endpoint, payload, *, accept_eula=False):
    """Invoke a typed SDK v3 Endpoint and decode its JSON response."""
    output = endpoint.invoke(
        body=json.dumps(payload),
        content_type="application/json",
        accept="application/json",
        custom_attributes="accept_eula=true" if accept_eula else None,
    )
    body = output.body.read() if hasattr(output.body, "read") else output.body
    if isinstance(body, bytes):
        body = body.decode("utf-8")
    return json.loads(body) if isinstance(body, str) else body


def generated_text(response):
    """Normalize common text-generation response envelopes."""
    if isinstance(response, list) and len(response) == 1:
        response = response[0]
    if not isinstance(response, dict) or "generated_text" not in response:
        raise ValueError(f"Unexpected endpoint response shape: {response!r}")
    return response["generated_text"]


def print_response(model_label, payload, response):
    print(f"Model: {model_label}")
    print(f"Prompt: {payload['inputs']}")
    print(f"Response: {generated_text(response)}")
    print("\n==================================\n")


payload = {
    "inputs": "### Instruction: What is Amazon SageMaker in one sentence?### Response:\n",
    "parameters": {
        "max_new_tokens": 128,
        "top_p": 0.9,
        "temperature": 0.6,
        "return_full_text": False,
    },
}

response = invoke_json(fine_tuned_model_endpoint, payload)
print_response(f"{model_id} (fine-tuned)", payload, response)


In [ ]:
import pandas as pd
from IPython.display import display, HTML

inputs, ground_truth_responses = [], []
responses_before_finetuning, responses_after_finetuning = [], []


def predict_and_print(datapoint):
    input_output_demarkation_key = "\n\n### Response:\n"
    payload = {
        "inputs": template["prompt"].format(
            instruction=datapoint["instruction"], context=datapoint["context"]
        ) + input_output_demarkation_key,
        "parameters": {"max_new_tokens": 100},
    }
    inputs.append(payload["inputs"])
    ground_truth_responses.append(datapoint["response"])

    pretrained_response = invoke_json(base_model_endpoint, payload, accept_eula=True)
    responses_before_finetuning.append(generated_text(pretrained_response))

    finetuned_response = invoke_json(fine_tuned_model_endpoint, payload)
    responses_after_finetuning.append(generated_text(finetuned_response))


for datapoint in test_dataset.select(range(min(5, test_dataset.num_rows))):
    predict_and_print(datapoint)

df = pd.DataFrame({
    "Inputs": inputs,
    "Ground Truth": ground_truth_responses,
    "Response from non-finetuned model": responses_before_finetuning,
    "Response from fine-tuned model": responses_after_finetuning,
})
display(HTML(df.to_html()))


In [ ]:
print(f"Experiment: {experiment_name}")
print(f"Run ID:     {mlflow_run.info.run_id}")


In [ ]:
# Persist the exact handoff consumed by Lab 3C.
run_id = mlflow_run.info.run_id
%store experiment_name
%store run_id
%store training_job_name
%store base_model_endpoint_name
%store fine_tuned_model_endpoint_name


## Step 9: Review Governance Artifacts in MLflow

### What We've Tracked

Throughout this lab, we've automatically logged:

1. **Data Lineage**
   - Source dataset location
   - Number of training/test examples
   - Data preprocessing steps

2. **Model Lineage**
   - Base model ID and version
   - All hyperparameters
   - Training job name
   - Model artifact location

3. **Deployment Lineage**
   - Endpoint container image
   - Instance type

### Accessing Your Experiments

You can view all tracked experiments in:
1. **SageMaker Studio**: Navigate to MLflow App
2. **MLflow UI**: Access through the MLflow App URL
3. **Programmatically**: Query using MLflow APIs

Since the training is now complete, let's mark the Run as completed.

In [ ]:
mlflow.end_run()

print("✓ MLflow run completed")
print("\nRun Summary:")
print(f"  Experiment:       {experiment_name}")
print(f"  Run ID:           {run_id}")
print(f"  Training job:     {training_job_name}")
print(f"  Model artifact:   {model_artifact_s3}")
print(f"  Inference image:  {inference_image_uri}")
print("\nAll parameters, metrics, artifacts, and deployment lineage are recorded.")


## Key Takeaways

In this lab, you learned how to:

1. ✅ **Fine-tune a foundation model** for a specific use case (text summarization)
2. ✅ **Track all experimentation** using SageMaker Managed MLflow
3. ✅ **Establish complete lineage** from data → training → deployment
4. ✅ **Create audit trails** for governance and compliance
5. ✅ **Enable reproducibility** by logging all parameters and artifacts

### Governance Benefits Demonstrated

- **Auditability**: Every training run is logged with complete metadata
- **Reproducibility**: Any experiment can be recreated from tracked parameters
- **Lineage**: Clear chain from source data to deployed model
- **Compliance**: Meet regulatory requirements for model documentation
- **Collaboration**: Team members can view and compare all experiments

### Next Steps

- In the next lab, you will evaluate your fine-tuned model, review the metrics, and register it to the SageMaker Model Registry.